# Global Topic Renaming (Part 2)

The per-cluster naming in `get_topic_names.ipynb` works locally: each topic is
named in isolation. This can produce **duplicate or ambiguous names** when two
clusters cover related sub-themes (e.g. both named "History of Synthetic Biology").

This notebook fixes that by giving the LLM a **global view** of all topics at
once. We use **OpenAI function calling** so the model returns a structured array
of `(topic_id, name)` pairs — one per cluster — guaranteeing distinct,
publication-ready names.

In [1]:
import os
import json
import yaml
import pandas as pd
from openai import OpenAI

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR = os.path.join('..', 'assets')
MODELS_DIR = os.path.join(ASSETS_DIR, 'topic_models')

MODEL = 'gpt-4.1-nano'

with open('prompts.yaml') as f:
    prompts = yaml.safe_load(f)

THEME = prompts['theme']
THEME_DESCRIPTION = prompts['theme_description']

# ── OpenAI client ─────────────────────────────────────────────────────────────
api_key = open('openai.key').read().strip()
client = OpenAI(api_key=api_key)

## 1. Load part-1 results

In [3]:
teams_names  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'),  sep='\t')
papers_names = pd.read_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t')

print(f'Teams:  {len(teams_names)} topics')
print(f'Papers: {len(papers_names)} topics')
teams_names[['topic', 'name', 'description']].head()

Teams:  103 topics
Papers: 234 topics


,topic,name,description
0,0,Environmental Bioremediation and Resource Reco...,This cluster centers on harnessing synthetic b...
1,1,Synthetic Biology-Enabled Disease Diagnostics,This collection of texts focuses on advancing ...
2,2,Light-Controlled Synthetic Biology Systems,This cluster focuses on the advancement and ut...
3,3,Synthetic Biology for Bacterial Quorum Sensing...,This cluster focuses on leveraging synthetic b...
4,4,Synthetic Biology for Plastic Biodegradation,This cluster centers on leveraging synthetic b...


## 2. Function-calling tool definition and renaming logic

In [4]:
def build_rename_tool(n_topics: int) -> dict:
    """Build the OpenAI function-calling tool schema for renaming n topics."""
    return {
        'type': 'function',
        'function': {
            'name': 'rename_topics',
            'description': (
                f'Assign a unique, concise, publication-ready name to each of '
                f'the {n_topics} topic clusters. Every name must be distinct.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'topics': {
                        'type': 'array',
                        'description': f'Exactly {n_topics} renamed topics.',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'topic_id': {
                                    'type': 'integer',
                                    'description': 'The original topic ID.',
                                },
                                'name': {
                                    'type': 'string',
                                    'description': (
                                        'A short, unique name for this topic '
                                        'Must be distinct from '
                                        'every other topic name.'
                                    ),
                                },
                            },
                            'required': ['topic_id', 'name'],
                        },
                    },
                },
                'required': ['topics'],
            },
        },
    }


def build_user_message(names_df: pd.DataFrame) -> str:
    """Format all topics into a single user message for the LLM."""
    lines = []
    for _, row in names_df.iterrows():
        lines.append(
            f"Topic {int(row['topic'])}:\n"
            f"  Current name: {row['name']}\n"
            f"  Description: {row['description']}\n"
        )
    return '\n'.join(lines)


SYSTEM_PROMPT = (
    f'You are a research consultant with expertise on {THEME}, meaning that '
    f'you know about {THEME_DESCRIPTION}\n\n'
    f'You will receive a list of topic clusters, each with an ID, a current '
    f'name, and a description. Some current names may be duplicated or '
    f'ambiguous. Your task is to assign a NEW, UNIQUE, short name to each '
    f'topic that:\n'
    f'1. Clearly distinguishes it from every other topic in the list.\n'
    f'2. Faithfully reflects the description.\n'
    f'3. Is concise and suitable for use in an academic publication.\n'
    f'4. No two names should overlap or be paraphrases of each other.\n\n'
    f'Return ALL topics, even if the current name is already good — confirm '
    f'or improve each one.'
)

In [5]:
def rename_topics_global(names_df: pd.DataFrame) -> pd.DataFrame:
    """Call OpenAI with function calling to get globally distinct topic names.

    Returns a copy of names_df with a new column `global_name`.
    """
    n = len(names_df)
    tool = build_rename_tool(n)
    user_msg = build_user_message(names_df)

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        tools=[tool],
        tool_choice={'type': 'function', 'function': {'name': 'rename_topics'}},
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
    )

    # Parse the function call arguments
    call = response.choices[0].message.tool_calls[0]
    args = json.loads(call.function.arguments)
    renamed = {item['topic_id']: item['name'] for item in args['topics']}

    result = names_df.copy()
    result['global_name'] = result['topic'].map(renamed)

    # Verify all topics got a name
    missing = result['global_name'].isna().sum()
    if missing > 0:
        print(f'WARNING: {missing} topic(s) did not receive a global name')

    # Verify uniqueness
    dupes = result['global_name'].duplicated().sum()
    if dupes > 0:
        print(f'WARNING: {dupes} duplicate global name(s) found')

    return result

## 3. Rename: iGEM Teams topics

In [6]:
teams_renamed = rename_topics_global(teams_names)
teams_renamed[['topic', 'name', 'global_name', 'description']]

,topic,name,global_name,description
0,0,Environmental Bioremediation and Resource Reco...,Environmental Bioremediation & Resource Recovery,This cluster centers on harnessing synthetic b...
1,1,Synthetic Biology-Enabled Disease Diagnostics,Synthetic Diagnostics & Biosensing,This collection of texts focuses on advancing ...
2,2,Light-Controlled Synthetic Biology Systems,Light-Regulated Synthetic Systems,This cluster focuses on the advancement and ut...
3,3,Synthetic Biology for Bacterial Quorum Sensing...,Synthetic Quorum Sensing & Biofilm Control,This cluster focuses on leveraging synthetic b...
4,4,Synthetic Biology for Plastic Biodegradation,Plastic Biodegradation & Recycling,This cluster centers on leveraging synthetic b...
...,...,...,...,...
98,98,Synthetic Biology for Pesticide Detection and ...,Pesticide Detection & Bioremediation,This cluster centers on leveraging genetic eng...
99,99,Synthetic Biology-Driven Antiviral Therapeutic...,Antiviral Synthetic Biology,This collection of texts focuses on harnessing...
100,100,Synthetic Biology for Biological Computing and...,Biological Computing & System Engineering,This collection of texts centers on leveraging...
101,101,Synthetic Biology for Advanced Biological Syst...,Advanced Biological Materials & Systems,This cluster centers on the innovative use of ...


## 4. Rename: SynBio Papers topics

In [7]:
papers_renamed = rename_topics_global(papers_names)
papers_renamed[['topic', 'name', 'global_name', 'description']]

,topic,name,global_name,description
0,0,Synthetic Vesicle-Based Artificial Cells,Artificial Vesicle-Based Cells,The focus of this cluster centers on the desig...
1,1,Governance and Responsible Innovation in Synth...,Governance and Responsible Innovation in Synth...,The cluster focuses on the governance and resp...
2,2,Plant-Derived Bioactive Compound Synthesis,Plant-Derived Bioactive Compound Biosynthesis,This cluster focuses on leveraging synthetic b...
3,3,Plant Synthetic Regulatory Elements,Synthetic Regulatory Elements in Plants,This cluster focuses on the development and ap...
4,4,Synthetic DNA Motifs for Immune Activation,Synthetic DNA Motifs for Immune Activation,This cluster focuses on the strategic use of s...
...,...,...,...,...
229,229,DNA Nanomachines,DNA Nanomachines and Molecular Devices,This cluster centers on the design and utiliza...
230,230,Synthetic Biology of Cannabinoids,Cannabinoid Biosynthesis,This cluster centers on the innovative manipul...
231,231,In Vitro Synthetic Biology,In Vitro Synthetic Life Systems,This cluster centers on the creation of in vit...
232,232,Mitochondrial Protein Engineering,Mitochondrial Protein Engineering,This collection of texts centers on utilizing ...


## 5. Save final results

In [8]:
# Overwrite the part-1 files with the global_name column added
teams_renamed.to_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'),  sep='\t', index=False)
papers_renamed.to_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t', index=False)

print(f'Saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

Saved to ../assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
